# Erlang C Calculator — Unit Tests

Place this notebook in the project's `tests` folder. Select the `Erlang C (.venv)` kernel. Run the setup cell first, then run each numbered test separately. These tests do not use the large CDR datasets.

## Notebook setup cell

In [1]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import Markdown, display

# Works when VS Code starts the notebook from either the project root or tests folder.
cwd = Path.cwd()
PROJECT_DIR = cwd if (cwd / "calculator.py").exists() else cwd.parent
if not (PROJECT_DIR / "calculator.py").exists():
    raise FileNotFoundError("Could not find calculator.py. Place this notebook inside the tests folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from calculator import (
    hms_to_seconds,
    clean_percent,
    calculate_traffic,
    average_speed_of_answer,
    required_agents,
    infer_single_year,
    build_cdr_intervals,
    build_shift_requirements,
    calculate_schedule_headcount,
)

UNIT_RESULTS = []

def record(test_id, description, passed, expected, actual, details=""):
    status = "PASS" if passed else "FAIL"
    UNIT_RESULTS.append({
        "test": test_id, "description": description, "status": status,
        "expected": str(expected), "actual": str(actual), "details": details,
    })
    print(f"{status}: {test_id} — {description}")
    print("Expected:", expected)
    print("Actual:  ", actual)
    if details: print("Details: ", details)
    return passed

def replace_previous(test_id):
    # Allows a cell to be rerun without counting the same test twice.
    UNIT_RESULTS[:] = [item for item in UNIT_RESULTS if item["test"] != test_id]

print("Setup complete")
print("Python:", sys.executable)
print("Project directory:", PROJECT_DIR)
print("calculator.py imported successfully")

Setup complete
Python: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Scripts\python.exe
Project directory: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C
calculator.py imported successfully


## Unit Test 1 — Checks whether the system correctly converts a call duration into seconds.

In [2]:
test_id = "UT-01"
replace_previous(test_id)
actual = hms_to_seconds("01:02:03")
expected = 3723
record(test_id, "Valid HH:MM:SS conversion", actual == expected, expected, actual)

PASS: UT-01 — Valid HH:MM:SS conversion
Expected: 3723
Actual:   3723


True

## Unit Test 2 — Checks whether incorrect time values are rejected.

In [3]:
test_id = "UT-02"
replace_previous(test_id)
inputs = ["01:60:00", "01:00:60", "invalid", "", None]
actual = {str(value): hms_to_seconds(value) for value in inputs}
passed = all(value is None for value in actual.values())
record(test_id, "Invalid durations return None", passed, "All results are None", actual)

PASS: UT-02 — Invalid durations return None
Expected: All results are None
Actual:   {'01:60:00': None, '01:00:60': None, 'invalid': None, '': None, 'None': None}


True

## Unit Test 3 — Convert percentages correctly

In [4]:
test_id = "UT-03"
replace_previous(test_id)
actual = {
    "80": clean_percent(80),
    "80%": clean_percent("80%"),
    "0.8": clean_percent(0.8),
    "30": clean_percent(30),
}
expected = {"80": 0.8, "80%": 0.8, "0.8": 0.8, "30": 0.3}
passed = all(abs(actual[k] - expected[k]) < 1e-9 for k in expected)
record(test_id, "Percentage normalization", passed, expected, actual)

PASS: UT-03 — Percentage normalization
Expected: {'80': 0.8, '80%': 0.8, '0.8': 0.8, '30': 0.3}
Actual:   {'80': 0.8, '80%': 0.8, '0.8': 0.8, '30': 0.3}


True

## Unit Test 4 — Checks whether the system correctly calculates call workload in Erlangs.

In [5]:
test_id = "UT-04"
replace_previous(test_id)
actual = calculate_traffic(call_volume=10, aht_seconds=60, interval_seconds=300)
expected = 2.0
record(test_id, "Traffic calculation", abs(actual - expected) < 1e-9, expected, actual, "(10 × 60) ÷ 300")

PASS: UT-04 — Traffic calculation
Expected: 2.0
Actual:   2.0
Details:  (10 × 60) ÷ 300


True

## Unit Test 5 — Reject invalid traffic inputs

In [7]:
test_id = "UT-05"
replace_previous(test_id)
cases = [
    ("negative calls", (-1, 60, 300)),
    ("zero AHT", (10, 0, 300)),
    ("zero interval", (10, 60, 0)),
]
outcomes = {}
for name, arguments in cases:
    try:
        calculate_traffic(*arguments)
        outcomes[name] = "No error"
    except ValueError as error:
        outcomes[name] = f"ValueError: {error}"
passed = all(value.startswith("ValueError:") for value in outcomes.values())
record(test_id, "Invalid traffic inputs raise ValueError", passed, "ValueError for all three cases", outcomes)

PASS: UT-05 — Invalid traffic inputs raise ValueError
Expected: ValueError for all three cases
Actual:   {'negative calls': 'ValueError: Call volume cannot be negative.', 'zero AHT': 'ValueError: AHT seconds must be greater than 0.', 'zero interval': 'ValueError: Interval seconds must be greater than 0.'}


True

## Unit Test 6 — Calculate Average Speed of Answer. It estimates how long a caller will wait before an agent answers.

In [8]:
test_id = "UT-06"
replace_previous(test_id)
actual = average_speed_of_answer(erlang_c=0.5, aht_seconds=300, agents=6, traffic=5)
expected = 150.0
record(test_id, "Average Speed of Answer calculation", abs(actual - expected) < 1e-9, expected, actual)

PASS: UT-06 — Average Speed of Answer calculation
Expected: 150.0
Actual:   150.0


True

## Unit Test 7 — Zero calls require zero agents. Checks what happens when no calls are expected.

In [11]:
test_id = "UT-07"
replace_previous(test_id)
result = required_agents(
    call_volume=0, aht_seconds=300, interval_seconds=1800,
    target_seconds=20, target_service_level=80, shrinkage=30,
)
actual = {"raw_agents": result["raw_agents"], "scheduled_agents": result["scheduled_agents"]}
expected = {"raw_agents": 0, "scheduled_agents": 0}
record(test_id, "Zero-call staffing", actual == expected, expected, actual)

PASS: UT-07 — Zero-call staffing
Expected: {'raw_agents': 0, 'scheduled_agents': 0}
Actual:   {'raw_agents': 0, 'scheduled_agents': 0}


True

## Unit Test 8 — Normal Erlang C result is sensible

In [12]:
test_id = "UT-08"
replace_previous(test_id)
result = required_agents(
    call_volume=30, aht_seconds=300, interval_seconds=1800,
    target_seconds=20, target_service_level=80, shrinkage=30,
)
checks = {
    "traffic_is_5": abs(result["traffic_erlangs"] - 5.0) < 1e-9,
    "raw_agents_positive": result["raw_agents"] > 0,
    "scheduled_not_less_than_raw": result["scheduled_agents"] >= result["raw_agents"],
    "service_level_between_0_and_1": 0 <= result["service_level"] <= 1,
    "occupancy_between_0_and_1": 0 <= result["occupancy"] <= 1,
}
record(test_id, "Normal Erlang C output properties", all(checks.values()), "All properties True", checks, str(result))

PASS: UT-08 — Normal Erlang C output properties
Expected: All properties True
Actual:   {'traffic_is_5': True, 'raw_agents_positive': True, 'scheduled_not_less_than_raw': True, 'service_level_between_0_and_1': True, 'occupancy_between_0_and_1': True}
Details:  {'traffic_erlangs': 5.0, 'raw_agents': 8, 'scheduled_agents': 12, 'service_level': 0.8630537670360967, 'probability_waiting': 0.16726650666175663, 'occupancy': 0.625, 'asa_seconds': 16.72665066617566}


True

## Unit Test 9 — Reject invalid staffing parameters

In [13]:
test_id = "UT-09"
replace_previous(test_id)
cases = {
    "negative_calls": dict(call_volume=-1, aht_seconds=300, interval_seconds=1800, target_seconds=20, target_service_level=80, shrinkage=30),
    "zero_aht": dict(call_volume=10, aht_seconds=0, interval_seconds=1800, target_seconds=20, target_service_level=80, shrinkage=30),
    "invalid_service_level": dict(call_volume=10, aht_seconds=300, interval_seconds=1800, target_seconds=20, target_service_level=100, shrinkage=30),
    "invalid_shrinkage": dict(call_volume=10, aht_seconds=300, interval_seconds=1800, target_seconds=20, target_service_level=80, shrinkage=100),
}
outcomes = {}
for name, parameters in cases.items():
    try:
        required_agents(**parameters)
        outcomes[name] = "No error"
    except ValueError as error:
        outcomes[name] = f"ValueError: {error}"
passed = all(value.startswith("ValueError:") for value in outcomes.values())
record(test_id, "Invalid staffing parameters raise ValueError", passed, "ValueError for all cases", outcomes)

PASS: UT-09 — Invalid staffing parameters raise ValueError
Expected: ValueError for all cases
Actual:   {'negative_calls': 'ValueError: Call volume cannot be negative.', 'zero_aht': 'ValueError: AHT seconds must be greater than 0.', 'invalid_service_level': 'ValueError: Target service level must be between 0% and 100%.', 'invalid_shrinkage': 'ValueError: Shrinkage must be between 0% and less than 100%.'}


True

## Unit Test 10 — Detect one calendar year

In [14]:
test_id = "UT-10"
replace_previous(test_id)
frame = pd.DataFrame({"call_datetime": pd.to_datetime(["2024-01-01", "2024-06-15", "2024-12-31"])})
actual = infer_single_year(frame, "sample.csv")
expected = 2024
record(test_id, "Single-year detection", actual == expected, expected, actual)

PASS: UT-10 — Single-year detection
Expected: 2024
Actual:   2024


True

## Unit Test 11 — Reject a dataset containing two years

In [15]:
test_id = "UT-11"
replace_previous(test_id)
frame = pd.DataFrame({"call_datetime": pd.to_datetime(["2023-12-31", "2024-01-01"])})
try:
    infer_single_year(frame, "mixed.csv")
    actual = "No error"
    passed = False
except ValueError as error:
    actual = f"ValueError: {error}"
    passed = True
record(test_id, "Mixed-year dataset rejection", passed, "ValueError", actual)

PASS: UT-11 — Mixed-year dataset rejection
Expected: ValueError
Actual:   ValueError: mixed.csv must contain exactly one calendar year; found [2023, 2024].


True

## Unit Test 12 — Group calls into 30-minute intervals

In [16]:
test_id = "UT-12"
replace_previous(test_id)
clean_data = pd.DataFrame({
    "call_datetime": pd.to_datetime(["2024-01-01 09:05:00", "2024-01-01 09:20:00", "2024-01-01 09:40:00"]),
    "duration_seconds": [60, 120, 180],
})
intervals = build_cdr_intervals(clean_data, interval_minutes=30, include_empty_intervals=False)
actual = {
    "interval_rows": len(intervals),
    "call_volumes": intervals["call_volume"].tolist(),
    "aht_seconds": intervals["aht_seconds"].astype(float).tolist(),
}
expected = {"interval_rows": 2, "call_volumes": [2, 1], "aht_seconds": [90.0, 180.0]}
record(test_id, "Thirty-minute interval aggregation", actual == expected, expected, actual)
display(intervals)

PASS: UT-12 — Thirty-minute interval aggregation
Expected: {'interval_rows': 2, 'call_volumes': [2, 1], 'aht_seconds': [90.0, 180.0]}
Actual:   {'interval_rows': 2, 'call_volumes': [2, 1], 'aht_seconds': [90.0, 180.0]}


,call_datetime,call_volume,total_handle_time_seconds,aht_seconds,interval_seconds
0,2024-01-01 09:00:00,2,180.0,90.0,1800
1,2024-01-01 09:30:00,1,180.0,180.0,1800


## Unit Test 13 — Reject an invalid interval length

In [17]:
test_id = "UT-13"
replace_previous(test_id)
clean_data = pd.DataFrame({
    "call_datetime": pd.to_datetime(["2024-01-01 09:00:00"]),
    "duration_seconds": [60],
})
try:
    build_cdr_intervals(clean_data, interval_minutes=17)
    actual = "No error"
    passed = False
except ValueError as error:
    actual = f"ValueError: {error}"
    passed = True
record(test_id, "Invalid interval rejection", passed, "ValueError", actual)

PASS: UT-13 — Invalid interval rejection
Expected: ValueError
Actual:   ValueError: interval_minutes must be a positive divisor of 1440.


True

## Unit Test 14 — Convert forecast intervals into three shifts

In [18]:
test_id = "UT-14"
replace_previous(test_id)
forecast = pd.DataFrame({
    "interval_start": pd.to_datetime([
        "2025-01-01 00:00:00", "2025-01-01 07:30:00",
        "2025-01-01 08:00:00", "2025-01-01 15:30:00",
        "2025-01-01 16:00:00", "2025-01-01 23:30:00",
    ]),
    "scheduled_agents": [2, 4, 3, 5, 2, 6],
})
requirements = build_shift_requirements(forecast, year=2025, month=1)
actual = dict(zip(requirements["shift_code"], requirements["required_agents"]))
expected = {"NIGHT": 4, "MORNING": 5, "EVENING": 6}
record(test_id, "Three-shift maximum requirements", actual == expected, expected, actual)
display(requirements)

PASS: UT-14 — Three-shift maximum requirements
Expected: {'NIGHT': 4, 'MORNING': 5, 'EVENING': 6}
Actual:   {'NIGHT': 4, 'MORNING': 5, 'EVENING': 6}


,date,weekday,shift_code,shift_name,shift_label,start_hour,end_hour,required_agents
0,2025-01-01,Wednesday,NIGHT,Night,00:00-08:00,0,8,4
1,2025-01-01,Wednesday,MORNING,Morning,08:00-16:00,8,16,5
2,2025-01-01,Wednesday,EVENING,Evening,16:00-00:00,16,24,6


## Unit Test 15 — Calculate minimum weekly headcount

In [19]:
test_id = "UT-15"
replace_previous(test_id)
requirements = pd.DataFrame({
    "date": pd.to_datetime(["2025-01-06"] * 3 + ["2025-01-07"] * 3),
    "required_agents": [2, 3, 2, 2, 3, 2],
})
actual = calculate_schedule_headcount(requirements, working_days_per_week=5)
expected = 7
record(test_id, "Minimum schedule headcount", actual == expected, expected, actual, "Daily requirement is 7, which is higher than ceil(14/5)=3.")

PASS: UT-15 — Minimum schedule headcount
Expected: 7
Actual:   7
Details:  Daily requirement is 7, which is higher than ceil(14/5)=3.


True

## Final unit-test report — run after Tests 1–15

In [20]:
expected_test_ids = [f"UT-{number:02d}" for number in range(1, 16)]
completed_ids = {item["test"] for item in UNIT_RESULTS}
not_run = [test_id for test_id in expected_test_ids if test_id not in completed_ids]
results_table = pd.DataFrame(UNIT_RESULTS).sort_values("test") if UNIT_RESULTS else pd.DataFrame()
passed = int((results_table["status"] == "PASS").sum()) if not results_table.empty else 0
failed = int((results_table["status"] == "FAIL").sum()) if not results_table.empty else 0
total = len(expected_test_ids)
rows = []
for test_id in expected_test_ids:
    found = next((item for item in UNIT_RESULTS if item["test"] == test_id), None)
    if found:
        rows.append(f"| {found['test']} | {found['description']} | {found['expected']} | {found['actual']} | {found['status']} |")
    else:
        rows.append(f"| {test_id} | Not run | — | — | NOT RUN |")
status = "PASS" if failed == 0 and not not_run else ("FAIL" if failed else "INCOMPLETE")
report = f"""
# Erlang C Unit Testing Report

## Test information

- Test date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
- Module tested: calculator.py
- Test type: Unit testing
- Python environment: {sys.executable}

## Overall results

| Result | Count |
|---|---:|
| Expected tests | {total} |
| Passed | {passed} |
| Failed | {failed} |
| Not run | {len(not_run)} |

## Detailed results

| Test | Description | Expected | Actual | Status |
|---|---|---|---|---|
{chr(10).join(rows)}

## Final status

{status}

{('All 15 unit tests passed.' if status == 'PASS' else ('Failed tests should be reviewed.' if status == 'FAIL' else 'Run the remaining test cells before finalizing the report.'))}
"""
display(Markdown(report))


# Erlang C Unit Testing Report

## Test information

- Test date: 2026-09-04 21:30
- Module tested: calculator.py
- Test type: Unit testing
- Python environment: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Scripts\python.exe

## Overall results

| Result | Count |
|---|---:|
| Expected tests | 15 |
| Passed | 15 |
| Failed | 0 |
| Not run | 0 |

## Detailed results

| Test | Description | Expected | Actual | Status |
|---|---|---|---|---|
| UT-01 | Valid HH:MM:SS conversion | 3723 | 3723 | PASS |
| UT-02 | Invalid durations return None | All results are None | {'01:60:00': None, '01:00:60': None, 'invalid': None, '': None, 'None': None} | PASS |
| UT-03 | Percentage normalization | {'80': 0.8, '80%': 0.8, '0.8': 0.8, '30': 0.3} | {'80': 0.8, '80%': 0.8, '0.8': 0.8, '30': 0.3} | PASS |
| UT-04 | Traffic calculation | 2.0 | 2.0 | PASS |
| UT-05 | Invalid traffic inputs raise ValueError | ValueError for all three cases | {'negative calls': 'ValueError: Call volume cannot be negative.', 'zero AHT': 'ValueError: AHT seconds must be greater than 0.', 'zero interval': 'ValueError: Interval seconds must be greater than 0.'} | PASS |
| UT-06 | Average Speed of Answer calculation | 150.0 | 150.0 | PASS |
| UT-07 | Zero-call staffing | {'raw_agents': 0, 'scheduled_agents': 0} | {'raw_agents': 0, 'scheduled_agents': 0} | PASS |
| UT-08 | Normal Erlang C output properties | All properties True | {'traffic_is_5': True, 'raw_agents_positive': True, 'scheduled_not_less_than_raw': True, 'service_level_between_0_and_1': True, 'occupancy_between_0_and_1': True} | PASS |
| UT-09 | Invalid staffing parameters raise ValueError | ValueError for all cases | {'negative_calls': 'ValueError: Call volume cannot be negative.', 'zero_aht': 'ValueError: AHT seconds must be greater than 0.', 'invalid_service_level': 'ValueError: Target service level must be between 0% and 100%.', 'invalid_shrinkage': 'ValueError: Shrinkage must be between 0% and less than 100%.'} | PASS |
| UT-10 | Single-year detection | 2024 | 2024 | PASS |
| UT-11 | Mixed-year dataset rejection | ValueError | ValueError: mixed.csv must contain exactly one calendar year; found [2023, 2024]. | PASS |
| UT-12 | Thirty-minute interval aggregation | {'interval_rows': 2, 'call_volumes': [2, 1], 'aht_seconds': [90.0, 180.0]} | {'interval_rows': 2, 'call_volumes': [2, 1], 'aht_seconds': [90.0, 180.0]} | PASS |
| UT-13 | Invalid interval rejection | ValueError | ValueError: interval_minutes must be a positive divisor of 1440. | PASS |
| UT-14 | Three-shift maximum requirements | {'NIGHT': 4, 'MORNING': 5, 'EVENING': 6} | {'NIGHT': 4, 'MORNING': 5, 'EVENING': 6} | PASS |
| UT-15 | Minimum schedule headcount | 7 | 7 | PASS |

## Final status

PASS

All 15 unit tests passed.
